In [2]:
import pandas as pd

Find uniqueness scores for job features - tech skills, tools, and tasks. 

Each feature will be rated on an inverted scale from 1 to 5 determined by the number of jobs that require them. The higher the score, the *fewer* jobs use it, and thus the greater uniqueness of the tech skill, tool, or task.

(Spent some time on this earlier but lost focus - will start over. Already have some aggregates put together.)

Skills, Abilities, Work Activities, Knowledge, and Work Context are already scaled 1 to 5 based on both frequency and importance. A mean will be taken for each of these.

All of these scores will be measured alongside with AI scores on my personal attributes.

# Tech Skills

In [25]:
jobs_agg_tech = pd.read_csv('transformed data/1-1-Jobs Agg Tech Skills.csv')
jobs_agg_tech

,O*NET-SOC Code,Job Title,TOTAL_TECH_SKILLS,TOTAL_TECH_TYPES,TECH_SKILLS_LIST,TECH_TYPES_LIST,PCT_HOT_TECH,NUM_HOT_TECH,PCT_IN_DEMAND_TECH,NUM_IN_DEMAND_TECH
0,15-1252.00,Software Developers,429,67,"""3M Post-it App""; ""A programming language APL""...","""Access software""; ""Administration software""; ...",34.498834,148,7.692308,33
1,15-1253.00,Software Quality Assurance Analysts and Testers,428,68,"""3M Post-it App""; ""A programming language APL""...","""Access software""; ""Accounting software""; ""Adm...",28.971963,124,5.140187,22
2,15-1299.09,Information Technology Project Managers,334,62,"""24SevenOffice Project""; ""3M Post-it App""; ""AE...","""Access software""; ""Accounting software""; ""Ana...",36.227545,121,2.694611,9
3,15-1243.00,Database Architects,324,59,"""3M Post-it App""; ""ADO.NET""; ""AJAX""; ""ASG Tech...","""Access software""; ""Administration software""; ...",34.567901,112,4.012346,13
4,15-1211.00,Computer Systems Analysts,313,68,"""3M Post-it App""; ""ADP Workforce Now""; ""AJAX"";...","""Access software""; ""Accounting software""; ""Adm...",38.019169,119,3.514377,11
...,...,...,...,...,...,...,...,...,...,...
918,35-9021.00,Dishwashers,2,2,"""Facebook""; ""Microsoft Windows""","""Operating system software""; ""Web page creatio...",100.000000,2,0.000000,0
919,53-7041.00,Hoist and Winch Operators,2,2,"""Microsoft Excel""; ""Microsoft Word""","""Spreadsheet software""; ""Word processing softw...",100.000000,2,0.000000,0
920,53-4041.00,Subway and Streetcar Operators,2,2,"""Microsoft Office software""; ""Word processing ...","""Office suite software""; ""Word processing soft...",50.000000,1,0.000000,0
921,51-4023.00,"Rolling Machine Setters, Operators, and Tender...",2,2,"""Email software""; ""Web browser software""","""Electronic mail software""; ""Internet browser ...",0.000000,0,0.000000,0


In [26]:
tech_agg_jobs = pd.read_csv('transformed data/1-1-Tech Skills Agg Jobs.csv')
tech_agg_jobs

,Tech Skill Title,Tech Type Title,Tech Type Code,Hot Technology,In Demand,TOTAL_JOBS_PER_TECH_SKILL,TOTAL_JOBS_PER_TECH_TYPE,JOB_TITLE_LIST
0,Adobe Acrobat,Document management software,43232202,1,0,165,293,"""Accountants and Auditors""; ""Administrative La..."
1,AdSense Tracker,Data base user interface and query software,43232306,0,0,2,632,"""Chief Executives""; ""Marketing Managers"""
2,Atlassian JIRA,Content workflow software,43232201,1,0,53,46,"""Administrative Services Managers""; ""Aerospace..."
3,Blackbaud The Raiser's Edge,Customer relationship management CRM software,43232303,0,0,44,155,"""Accountants and Auditors""; ""Administrative Se..."
4,ComputerEase construction accounting software,Accounting software,43231601,0,0,3,178,"""Bookkeeping, Accounting, and Auditing Clerks""..."
...,...,...,...,...,...,...,...,...
8780,AMCS Platform,Analytical or scientific software,43232605,0,0,1,372,"""Refuse and Recyclable Material Collectors"""
8781,Dossier software,Data base user interface and query software,43232306,0,0,1,632,"""Refuse and Recyclable Material Collectors"""
8782,Mileage logging software,Data base user interface and query software,43232306,0,0,1,632,"""Refuse and Recyclable Material Collectors"""
8783,Routeware software,Map creation software,43233506,0,0,1,87,"""Refuse and Recyclable Material Collectors"""


In [27]:
jobs_agg_tech.columns

Index(['O*NET-SOC Code', 'Job Title', 'TOTAL_TECH_SKILLS', 'TOTAL_TECH_TYPES',
       'TECH_SKILLS_LIST', 'TECH_TYPES_LIST', 'PCT_HOT_TECH', 'NUM_HOT_TECH',
       'PCT_IN_DEMAND_TECH', 'NUM_IN_DEMAND_TECH'],
      dtype='object')

In [29]:
tech_agg_jobs.columns

Index(['Tech Skill Title', 'Tech Type Title', 'Tech Type Code',
       'Hot Technology', 'In Demand', 'TOTAL_JOBS_PER_TECH_SKILL',
       'TOTAL_JOBS_PER_TECH_TYPE', 'JOB_TITLE_LIST'],
      dtype='object')

In [30]:
import pandas as pd
import numpy as np

# ----------------------------
# Helpers
# ----------------------------
def _to_list(x):
    """Accepts list-like or delimited string like '"A"; "B"' and returns a list of items."""
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return [str(i).strip() for i in x if str(i).strip()]
    s = str(x).strip()
    if not s:
        return []
    # split on semicolon
    parts = [p.strip() for p in s.split(";")]
    # strip surrounding quotes
    parts = [p.strip().strip('"').strip("'") for p in parts]
    return [p for p in parts if p]

def add_uniqueness_score(df, name_col, count_col, score_col, q=5):
    """
    Creates a 1..q uniqueness score where lower counts => higher uniqueness.
    Uses quantile bins (qcut) for robustness with skew.
    """
    out = df[[name_col, count_col]].drop_duplicates().copy()

    # rank(method='first') avoids qcut issues with many ties
    ranks = out[count_col].rank(method="first")
    out[score_col] = (q - pd.qcut(ranks, q=q, labels=False)).astype(int)  # q..1
    return out

# ----------------------------
# 1) Tech skill uniqueness (1–5)
# ----------------------------
tech_skill_scores = add_uniqueness_score(
    df=tech_agg_jobs,
    name_col="Tech Skill Title",
    count_col="TOTAL_JOBS_PER_TECH_SKILL",
    score_col="tech_skill_uniqueness_1to5",
    q=5
)

# Optional 0–1 version
tech_skill_scores["tech_skill_uniqueness_0to1"] = (tech_skill_scores["tech_skill_uniqueness_1to5"] - 1) / 4

# ----------------------------
# 2) Tech type uniqueness (1–5)  (optional but usually useful)
# ----------------------------
tech_type_scores = add_uniqueness_score(
    df=tech_agg_jobs,
    name_col="Tech Type Title",
    count_col="TOTAL_JOBS_PER_TECH_TYPE",
    score_col="tech_type_uniqueness_1to5",
    q=5
).drop_duplicates(subset=["Tech Type Title"])

tech_type_scores["tech_type_uniqueness_0to1"] = (tech_type_scores["tech_type_uniqueness_1to5"] - 1) / 4

# ----------------------------
# 3) Job-level averages
# ----------------------------
# Expand TECH_SKILLS_LIST to one row per (job, tech skill)
job_skill_long = (
    jobs_agg_tech[["O*NET-SOC Code", "Job Title", "TECH_SKILLS_LIST"]]
    .assign(Tech_Skill_Title=lambda d: d["TECH_SKILLS_LIST"].apply(_to_list))
    .explode("Tech_Skill_Title")
    .drop(columns=["TECH_SKILLS_LIST"])
    .rename(columns={"Tech_Skill_Title": "Tech Skill Title"})
)

# Merge scores and aggregate
job_skill_scored = job_skill_long.merge(
    tech_skill_scores,
    on="Tech Skill Title",
    how="left"
)

job_skill_uniqueness = (
    job_skill_scored
    .groupby(["O*NET-SOC Code", "Job Title"], as_index=False)
    .agg(
        AVG_TECH_SKILL_UNIQUENESS_1to5=("tech_skill_uniqueness_1to5", "mean"),
        AVG_TECH_SKILL_UNIQUENESS_0to1=("tech_skill_uniqueness_0to1", "mean"),
        NUM_TECH_SKILLS=("Tech Skill Title", "nunique"),
    )
)

# Expand TECH_TYPES_LIST to one row per (job, tech type)
job_type_long = (
    jobs_agg_tech[["O*NET-SOC Code", "Job Title", "TECH_TYPES_LIST"]]
    .assign(Tech_Type_Title=lambda d: d["TECH_TYPES_LIST"].apply(_to_list))
    .explode("Tech_Type_Title")
    .drop(columns=["TECH_TYPES_LIST"])
    .rename(columns={"Tech_Type_Title": "Tech Type Title"})
)

job_type_scored = job_type_long.merge(
    tech_type_scores,
    on="Tech Type Title",
    how="left"
)

job_type_uniqueness = (
    job_type_scored
    .groupby(["O*NET-SOC Code", "Job Title"], as_index=False)
    .agg(
        AVG_TECH_TYPE_UNIQUENESS_1to5=("tech_type_uniqueness_1to5", "mean"),
        AVG_TECH_TYPE_UNIQUENESS_0to1=("tech_type_uniqueness_0to1", "mean"),
        NUM_TECH_TYPES=("Tech Type Title", "nunique"),
    )
)

# ----------------------------
# 4) Final combined dataframe (like your job_tool_uniqueness)
# ----------------------------
job_tech_uniqueness = (
    job_skill_uniqueness
    .merge(job_type_uniqueness, on=["O*NET-SOC Code", "Job Title"], how="outer")
)

# Optional: quick checks for mismatches (titles present in jobs list but not in agg table)
missing_skill_titles = job_skill_scored.loc[
    job_skill_scored["tech_skill_uniqueness_1to5"].isna(), "Tech Skill Title"
].dropna().drop_duplicates()

missing_type_titles = job_type_scored.loc[
    job_type_scored["tech_type_uniqueness_1to5"].isna(), "Tech Type Title"
].dropna().drop_duplicates()

job_tech_uniqueness

,O*NET-SOC Code,Job Title,AVG_TECH_SKILL_UNIQUENESS_1to5,AVG_TECH_SKILL_UNIQUENESS_0to1,NUM_TECH_SKILLS,AVG_TECH_TYPE_UNIQUENESS_1to5,AVG_TECH_TYPE_UNIQUENESS_0to1,NUM_TECH_TYPES
0,11-1011.00,Chief Executives,1.285714,0.071429,49,1.347826,0.086957,23
1,11-1011.03,Chief Sustainability Officers,1.000000,0.000000,21,1.125000,0.031250,16
2,11-1021.00,General and Operations Managers,1.246575,0.061644,146,1.732143,0.183036,56
3,11-1031.00,Legislators,2.406250,0.351562,32,1.500000,0.125000,16
4,11-2011.00,Advertising and Promotions Managers,1.397260,0.099315,73,1.296296,0.074074,27
...,...,...,...,...,...,...,...,...
918,53-7071.00,Gas Compressor and Gas Pumping Station Operators,1.000000,0.000000,6,1.333333,0.083333,6
919,53-7072.00,"Pump Operators, Except Wellhead Pumpers",1.000000,0.000000,7,1.285714,0.071429,7
920,53-7073.00,Wellhead Pumpers,1.125000,0.031250,8,1.142857,0.035714,7
921,53-7081.00,Refuse and Recyclable Material Collectors,1.400000,0.100000,10,2.000000,0.250000,9


# Tools

In [13]:
tools_agg_jobs = pd.read_csv('transformed data/1-2-Tools Agg Jobs.csv')
tools_agg_jobs

,Tool Name,TOTAL_JOBS_BY_TOOL,JOBS_LIST
0,Personal computers,656,"""Accountants and Auditors""; ""Actors""; ""Actuari..."
1,Desktop computers,570,"""Accountants and Auditors""; ""Actuaries""; ""Acut..."
2,Laptop computers,425,"""Accountants and Auditors""; ""Actors""; ""Acute C..."
3,Tablet computers,227,"""Accountants and Auditors""; ""Actors""; ""Acute C..."
4,Laser facsimile machines,166,"""Accountants and Auditors""; ""Administrative La..."
...,...,...,...
18100,Half-round hand files,1,"""Craft Artists"""
18101,Half-round needle files,1,"""Craft Artists"""
18102,Half-round pliers,1,"""Craft Artists"""
18103,Halide meters,1,"""Environmental Science and Protection Technici..."


In [14]:
jobs_agg_tools = pd.read_csv('transformed data/1-2-Jobs Agg Tools.csv')
jobs_agg_tools

,O*NET-SOC Code,Job Title,Tool Name,Tool Type Code,Tool Type Title,TOTAL_TOOLS_BY_JOB,TOTAL_TOOL_TYPES_BY_JOB,TOOLS_LIST,TOOL_TYPES_LIST
0,11-1011.00,Chief Executives,10-key calculators,44101809,Desktop calculator,7,7,"""10-key calculators""; ""Desktop computers""; ""La...","""Desktop calculator""; ""Desktop computer""; ""Hig..."
1,11-1011.00,Chief Executives,Desktop computers,43211507,Desktop computer,7,7,"""10-key calculators""; ""Desktop computers""; ""La...","""Desktop calculator""; ""Desktop computer""; ""Hig..."
2,11-1011.00,Chief Executives,Laptop computers,43211503,Notebook computer,7,7,"""10-key calculators""; ""Desktop computers""; ""La...","""Desktop calculator""; ""Desktop computer""; ""Hig..."
3,11-1011.00,Chief Executives,Personal computers,43211508,Personal computer,7,7,"""10-key calculators""; ""Desktop computers""; ""La...","""Desktop calculator""; ""Desktop computer""; ""Hig..."
4,11-1011.00,Chief Executives,Personal digital assistants PDA,43211504,Personal digital assistant PDAs or organizer,7,7,"""10-key calculators""; ""Desktop computers""; ""La...","""Desktop calculator""; ""Desktop computer""; ""Hig..."
...,...,...,...,...,...,...,...,...,...
41657,53-7121.00,"Tank Car, Truck, and Ship Loaders",Stock pickers,24101605,Loading equipment,17,15,"""Computer laser printers""; ""Conveyor belt syst...","""Belt conveyors""; ""Floor or platform scales""; ..."
41658,53-7121.00,"Tank Car, Truck, and Ship Loaders",Tablet computers,43211509,Tablet computer,17,15,"""Computer laser printers""; ""Conveyor belt syst...","""Belt conveyors""; ""Floor or platform scales""; ..."
41659,53-7121.00,"Tank Car, Truck, and Ship Loaders",Walk behind pallet jacks,24101505,Pallet trucks,17,15,"""Computer laser printers""; ""Conveyor belt syst...","""Belt conveyors""; ""Floor or platform scales""; ..."
41660,53-7121.00,"Tank Car, Truck, and Ship Loaders",Warehouse forklifts,24101603,Forklifts,17,15,"""Computer laser printers""; ""Conveyor belt syst...","""Belt conveyors""; ""Floor or platform scales""; ..."


In [20]:
tools_agg_jobs.columns

Index(['Tool Name', 'TOTAL_JOBS_BY_TOOL', 'JOBS_LIST'], dtype='object')

In [21]:
jobs_agg_tools.columns

Index(['O*NET-SOC Code', 'Job Title', 'Tool Name', 'Tool Type Code',
       'Tool Type Title', 'TOTAL_TOOLS_BY_JOB', 'TOTAL_TOOL_TYPES_BY_JOB',
       'TOOLS_LIST', 'TOOL_TYPES_LIST'],
      dtype='object')

In [19]:
df_tools_used.columns

Index(['O*NET-SOC Code', 'Title', 'Example', 'Commodity Code',
       'Commodity Title'],
      dtype='object')

In [22]:
import pandas as pd
import numpy as np

# ----------------------------
# 1) Build per-tool uniqueness score (1–5)
# ----------------------------
# Higher TOTAL_JOBS_BY_TOOL => more common => LOWER uniqueness score
tool_scores = tools_agg_jobs[['Tool Name', 'TOTAL_JOBS_BY_TOOL']].copy()

# Use quantiles to bin into 5 roughly-equal groups.
# qcut gives 0..4, then we invert so 5=most unique, 1=least unique.
tool_scores['tool_uniqueness_score_1to5'] = (
    5 - pd.qcut(
        tool_scores['TOTAL_JOBS_BY_TOOL'].rank(method='first'),
        q=5,
        labels=False
    )
).astype(int)

# Optional: also create a 0–1 version if you ever want it later
tool_scores['tool_uniqueness_0to1'] = (tool_scores['tool_uniqueness_score_1to5'] - 1) / 4

# ----------------------------
# 2) Attach score to the job↔tool rows
# ----------------------------
job_tools_scored = jobs_agg_tools.merge(
    tool_scores[['Tool Name', 'tool_uniqueness_score_1to5', 'tool_uniqueness_0to1']],
    on='Tool Name',
    how='left'
)

# (Quick sanity check: tools in jobs_agg_tools missing from tools_agg_jobs)
missing_tools = job_tools_scored.loc[
    job_tools_scored['tool_uniqueness_score_1to5'].isna(),
    'Tool Name'
].drop_duplicates()

# ----------------------------
# 3) Aggregate to job level: average tool uniqueness per job
# ----------------------------
job_tool_uniqueness = (
    job_tools_scored
    .groupby(['O*NET-SOC Code', 'Job Title'], as_index=False)
    .agg(
        AVG_TOOL_UNIQUENESS_1to5=('tool_uniqueness_score_1to5', 'mean'),
        AVG_TOOL_UNIQUENESS_0to1=('tool_uniqueness_0to1', 'mean'),
        NUM_TOOLS=('Tool Name', 'nunique')
    )
)

# Optional: round the averages
job_tool_uniqueness['AVG_TOOL_UNIQUENESS_1to5'] = job_tool_uniqueness['AVG_TOOL_UNIQUENESS_1to5'].round(3)
job_tool_uniqueness['AVG_TOOL_UNIQUENESS_0to1'] = job_tool_uniqueness['AVG_TOOL_UNIQUENESS_0to1'].round(3)

job_tool_uniqueness

,O*NET-SOC Code,Job Title,AVG_TOOL_UNIQUENESS_1to5,AVG_TOOL_UNIQUENESS_0to1,NUM_TOOLS
0,11-1011.00,Chief Executives,1.000,0.000,7
1,11-1011.03,Chief Sustainability Officers,1.000,0.000,10
2,11-1021.00,General and Operations Managers,1.278,0.069,18
3,11-1031.00,Legislators,2.000,0.250,4
4,11-2011.00,Advertising and Promotions Managers,1.000,0.000,12
...,...,...,...,...,...
897,53-7071.00,Gas Compressor and Gas Pumping Station Operators,1.647,0.162,17
898,53-7072.00,"Pump Operators, Except Wellhead Pumpers",2.625,0.406,40
899,53-7073.00,Wellhead Pumpers,2.768,0.442,69
900,53-7081.00,Refuse and Recyclable Material Collectors,2.812,0.453,16


# Tasks

In [35]:
df_task_statements

,O*NET-SOC Code,Title,Task ID,Task,Task Type,Incumbents Responding,Date,Domain Source
0,11-1011.00,Chief Executives,8823,Direct or coordinate an organization's financi...,Core,95.0,08/2023,Incumbent
1,11-1011.00,Chief Executives,8824,"Confer with board members, organization offici...",Core,95.0,08/2023,Incumbent
2,11-1011.00,Chief Executives,8827,"Prepare budgets for approval, including those ...",Core,95.0,08/2023,Incumbent
3,11-1011.00,Chief Executives,8826,"Direct, plan, or implement policies, objective...",Core,94.0,08/2023,Incumbent
4,11-1011.00,Chief Executives,8834,Prepare or present reports concerning activiti...,Core,95.0,08/2023,Incumbent
...,...,...,...,...,...,...,...,...
18791,53-7121.00,"Tank Car, Truck, and Ship Loaders",12807,Unload cars containing liquids by connecting h...,Supplemental,85.0,08/2019,Incumbent
18792,53-7121.00,"Tank Car, Truck, and Ship Loaders",12804,"Clean interiors of tank cars or tank trucks, u...",Supplemental,85.0,08/2019,Incumbent
18793,53-7121.00,"Tank Car, Truck, and Ship Loaders",12803,Lower gauge rods into tanks or read meters to ...,Supplemental,85.0,08/2019,Incumbent
18794,53-7121.00,"Tank Car, Truck, and Ship Loaders",12805,Operate conveyors and equipment to transfer gr...,Supplemental,85.0,08/2019,Incumbent


In [16]:
tasks_agg_jobs = pd.read_csv('transformed data/1-3-Tasks Agg Jobs.csv')
tasks_agg_jobs

,O*NET-SOC Code,Title,Task ID,Task,Task Type,Incumbents Responding,Date,Domain Source,Task_clean,Task_Concept_ID,Concept_Instance_Number,Semantic_Instance_ID,Task_String_ID,TOTAL_JOBS_BY_TASK,JOB_LIST
0,11-1011.00,Chief Executives,8823,Direct or coordinate an organization's financi...,Core,95.0,08/2023,Incumbent,Direct or coordinate an organization's financi...,5388,1,5388-1,1,1,"""Chief Executives"""
1,11-1011.00,Chief Executives,8824,"Confer with board members, organization offici...",Core,95.0,08/2023,Incumbent,"Confer with board members, organization offici...",3092,1,3092-1,2,1,"""Chief Executives"""
2,11-1011.00,Chief Executives,8827,"Prepare budgets for approval, including those ...",Core,95.0,08/2023,Incumbent,"Prepare budgets for approval, including those ...",12105,1,12105-1,3,1,"""Chief Executives"""
3,11-1011.00,Chief Executives,8826,"Direct, plan, or implement policies, objective...",Core,94.0,08/2023,Incumbent,"Direct, plan, or implement policies, objective...",5504,1,5504-1,4,1,"""Chief Executives"""
4,11-1011.00,Chief Executives,8834,Prepare or present reports concerning activiti...,Core,95.0,08/2023,Incumbent,Prepare or present reports concerning activiti...,12319,1,12319-1,5,1,"""Chief Executives"""
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18791,53-7121.00,"Tank Car, Truck, and Ship Loaders",12807,Unload cars containing liquids by connecting h...,Supplemental,85.0,08/2019,Incumbent,Unload cars containing liquids by connecting h...,17006,1,17006-1,17533,1,"""Tank Car, Truck, and Ship Loaders"""
18792,53-7121.00,"Tank Car, Truck, and Ship Loaders",12804,"Clean interiors of tank cars or tank trucks, u...",Supplemental,85.0,08/2019,Incumbent,"Clean interiors of tank cars or tank trucks, u...",1975,1,1975-1,17534,1,"""Tank Car, Truck, and Ship Loaders"""
18793,53-7121.00,"Tank Car, Truck, and Ship Loaders",12803,Lower gauge rods into tanks or read meters to ...,Supplemental,85.0,08/2019,Incumbent,Lower gauge rods into tanks or read meters to ...,8769,1,8769-1,17535,1,"""Tank Car, Truck, and Ship Loaders"""
18794,53-7121.00,"Tank Car, Truck, and Ship Loaders",12805,Operate conveyors and equipment to transfer gr...,Supplemental,85.0,08/2019,Incumbent,Operate conveyors and equipment to transfer gr...,10488,1,10488-1,17536,1,"""Tank Car, Truck, and Ship Loaders"""


In [17]:
jobs_agg_tasks = pd.read_csv('transformed data/1-3-Jobs Agg Tasks.csv')
jobs_agg_tasks

,O*NET-SOC Code,Title,TOTAL_TASKS_BY_JOB,TASK_LIST
0,25-2058.00,"Special Education Teachers, Secondary School",40,"""Administer standardized ability and achieveme..."
1,25-2057.00,"Special Education Teachers, Middle School",40,"""Administer standardized ability and achieveme..."
2,49-2022.00,Telecommunications Equipment Installers and Re...,39,"""Address special issues or situations, such as..."
3,19-2041.03,Industrial Ecologists,38,"""Analyze changes designed to improve the envir..."
4,25-2021.00,"Elementary School Teachers, Except Special Edu...",38,"""Adapt teaching methods and instructional mate..."
...,...,...,...,...
918,53-7031.00,Dredge Operators,6,"""Direct or assist workers placing shore anchor..."
919,53-1041.00,Aircraft Cargo Handling Supervisors,6,"""Accompany aircraft as a member of the flight ..."
920,47-2072.00,Pile Driver Operators,5,"""Clean, lubricate, and refill equipment.""; ""Co..."
921,45-2041.00,"Graders and Sorters, Agricultural Products",5,"""Discard inferior or defective products or for..."


In [31]:
tasks_agg_jobs.columns

Index(['O*NET-SOC Code', 'Title', 'Task ID', 'Task', 'Task Type',
       'Incumbents Responding', 'Date', 'Domain Source', 'Task_clean',
       'Task_Concept_ID', 'Concept_Instance_Number', 'Semantic_Instance_ID',
       'Task_String_ID', 'TOTAL_JOBS_BY_TASK', 'JOB_LIST'],
      dtype='object')

In [32]:
jobs_agg_tasks.columns

Index(['O*NET-SOC Code', 'Title', 'TOTAL_TASKS_BY_JOB', 'TASK_LIST'], dtype='object')

In [44]:
import pandas as pd
import numpy as np

def add_uniqueness_score(df, id_col, count_col, score_col, q=5):
    out = df[[id_col, count_col]].drop_duplicates().copy()
    ranks = out[count_col].rank(method="first")
    out[score_col] = (q - pd.qcut(ranks, q=q, labels=False)).astype(int)  # q..1
    return out

# 1) Build concept-level job counts from the raw task-by-job rows
concept_counts = (
    tasks_agg_jobs
    .dropna(subset=["Task_Concept_ID"])
    .groupby("Task_Concept_ID", as_index=False)
    .agg(TOTAL_JOBS_BY_CONCEPT=("O*NET-SOC Code", "nunique"))
)

# 2) Convert to 1–5 uniqueness score (lower job count => higher uniqueness)
concept_scores = add_uniqueness_score(
    concept_counts,
    id_col="Task_Concept_ID",
    count_col="TOTAL_JOBS_BY_CONCEPT",
    score_col="task_concept_uniqueness_1to5",
    q=5
)
concept_scores["task_concept_uniqueness_0to1"] = (concept_scores["task_concept_uniqueness_1to5"] - 1) / 4

# 3) Attach to each task row then average by job
tasks_scored = tasks_agg_jobs.merge(concept_scores, on="Task_Concept_ID", how="left")

job_task_uniqueness = (
    tasks_scored
    .groupby(["O*NET-SOC Code", "Title"], as_index=False)
    .agg(
        AVG_TASK_UNIQUENESS_1to5=("task_concept_uniqueness_1to5", "mean"),
        AVG_TASK_UNIQUENESS_0to1=("task_concept_uniqueness_0to1", "mean"),
        NUM_TASK_CONCEPTS=("Task_Concept_ID", "nunique"),
        NUM_TASK_ROWS=("Task_Concept_ID", "size"),
    )
)

job_task_uniqueness["AVG_TASK_UNIQUENESS_1to5"] = job_task_uniqueness["AVG_TASK_UNIQUENESS_1to5"].round(3)
job_task_uniqueness["AVG_TASK_UNIQUENESS_0to1"] = job_task_uniqueness["AVG_TASK_UNIQUENESS_0to1"].round(3)

job_task_uniqueness.rename(columns={'Title': 'Job Title'}, inplace=True)

job_task_uniqueness

,O*NET-SOC Code,Job Title,AVG_TASK_UNIQUENESS_1to5,AVG_TASK_UNIQUENESS_0to1,NUM_TASK_CONCEPTS,NUM_TASK_ROWS
0,11-1011.00,Chief Executives,3.290,0.573,31,31
1,11-1011.03,Chief Sustainability Officers,3.056,0.514,18,18
2,11-1021.00,General and Operations Managers,2.765,0.441,17,17
3,11-1031.00,Legislators,3.000,0.500,30,30
4,11-2011.00,Advertising and Promotions Managers,3.400,0.600,30,30
...,...,...,...,...,...,...
918,53-7071.00,Gas Compressor and Gas Pumping Station Operators,2.692,0.423,13,13
919,53-7072.00,"Pump Operators, Except Wellhead Pumpers",2.857,0.464,14,14
920,53-7073.00,Wellhead Pumpers,2.714,0.429,14,14
921,53-7081.00,Refuse and Recyclable Material Collectors,3.429,0.607,14,14


# Merging all

In [38]:
job_tech_uniqueness.head(5)

,O*NET-SOC Code,Job Title,AVG_TECH_SKILL_UNIQUENESS_1to5,AVG_TECH_SKILL_UNIQUENESS_0to1,NUM_TECH_SKILLS,AVG_TECH_TYPE_UNIQUENESS_1to5,AVG_TECH_TYPE_UNIQUENESS_0to1,NUM_TECH_TYPES
0,11-1011.00,Chief Executives,1.285714,0.071429,49,1.347826,0.086957,23
1,11-1011.03,Chief Sustainability Officers,1.000000,0.000000,21,1.125000,0.031250,16
2,11-1021.00,General and Operations Managers,1.246575,0.061644,146,1.732143,0.183036,56
3,11-1031.00,Legislators,2.406250,0.351562,32,1.500000,0.125000,16
4,11-2011.00,Advertising and Promotions Managers,1.397260,0.099315,73,1.296296,0.074074,27


In [39]:
job_tool_uniqueness.head(5)

,O*NET-SOC Code,Job Title,AVG_TOOL_UNIQUENESS_1to5,AVG_TOOL_UNIQUENESS_0to1,NUM_TOOLS
0,11-1011.00,Chief Executives,1.000,0.000,7
1,11-1011.03,Chief Sustainability Officers,1.000,0.000,10
2,11-1021.00,General and Operations Managers,1.278,0.069,18
3,11-1031.00,Legislators,2.000,0.250,4
4,11-2011.00,Advertising and Promotions Managers,1.000,0.000,12


In [45]:
job_task_uniqueness.head(5)

,O*NET-SOC Code,Job Title,AVG_TASK_UNIQUENESS_1to5,AVG_TASK_UNIQUENESS_0to1,NUM_TASK_CONCEPTS,NUM_TASK_ROWS
0,11-1011.00,Chief Executives,3.290,0.573,31,31
1,11-1011.03,Chief Sustainability Officers,3.056,0.514,18,18
2,11-1021.00,General and Operations Managers,2.765,0.441,17,17
3,11-1031.00,Legislators,3.000,0.500,30,30
4,11-2011.00,Advertising and Promotions Managers,3.400,0.600,30,30


In [52]:
merge_1 = pd.merge(job_tech_uniqueness, job_tool_uniqueness, on=['O*NET-SOC Code', 'Job Title'], how='inner') # 'how' can be 'inner', 'outer', 'left', or 'right'
final_merged_df = pd.merge(merge_1, job_task_uniqueness, on=['O*NET-SOC Code', 'Job Title'], how='inner')

# Keep only the 1 to 5 scale
# string_to_find = '_0to1'
# final_merged_df = final_merged_df.filter(regex=f'(?!.*{string_to_find}).*')

# v2
string_to_find = '_0to1'
columns_to_drop = final_merged_df.columns[final_merged_df.columns.str.contains(string_to_find)]
final_merged_df = final_merged_df.drop(columns=columns_to_drop)

final_merged_df

,O*NET-SOC Code,Job Title,AVG_TECH_SKILL_UNIQUENESS_1to5,NUM_TECH_SKILLS,AVG_TECH_TYPE_UNIQUENESS_1to5,NUM_TECH_TYPES,AVG_TOOL_UNIQUENESS_1to5,NUM_TOOLS,AVG_TASK_UNIQUENESS_1to5,NUM_TASK_CONCEPTS,NUM_TASK_ROWS
0,11-1011.00,Chief Executives,1.285714,49,1.347826,23,1.000,7,3.290,31,31
1,11-1011.03,Chief Sustainability Officers,1.000000,21,1.125000,16,1.000,10,3.056,18,18
2,11-1021.00,General and Operations Managers,1.246575,146,1.732143,56,1.278,18,2.765,17,17
3,11-1031.00,Legislators,2.406250,32,1.500000,16,2.000,4,3.000,30,30
4,11-2011.00,Advertising and Promotions Managers,1.397260,73,1.296296,27,1.000,12,3.400,30,30
...,...,...,...,...,...,...,...,...,...,...,...
897,53-7071.00,Gas Compressor and Gas Pumping Station Operators,1.000000,6,1.333333,6,1.647,17,2.692,13,13
898,53-7072.00,"Pump Operators, Except Wellhead Pumpers",1.000000,7,1.285714,7,2.625,40,2.857,14,14
899,53-7073.00,Wellhead Pumpers,1.125000,8,1.142857,7,2.768,69,2.714,14,14
900,53-7081.00,Refuse and Recyclable Material Collectors,1.400000,10,2.000000,9,2.812,16,3.429,14,14


In [53]:
final_merged_df.columns

Index(['O*NET-SOC Code', 'Job Title', 'AVG_TECH_SKILL_UNIQUENESS_1to5',
       'NUM_TECH_SKILLS', 'AVG_TECH_TYPE_UNIQUENESS_1to5', 'NUM_TECH_TYPES',
       'AVG_TOOL_UNIQUENESS_1to5', 'NUM_TOOLS', 'AVG_TASK_UNIQUENESS_1to5',
       'NUM_TASK_CONCEPTS', 'NUM_TASK_ROWS'],
      dtype='object')

In [55]:
final_merged_df.to_csv('transformed data/2-Job Uniqueness by Tech Tasks and Tools.csv', index=False)